# HelicAI

### Predict and design the chiroptical (ECD) spectrum of [6]helicene derivatives with a neural network

Welcome! This notebook lets you do two things, without writing any code yourself:

- **Block I — Predict your helicene.** Give the notebook a set of substituents and positions (or even a *brand-new* substituent the model has never seen before) and get back a predicted ECD spectrum in seconds, instead of hours of quantum-chemistry calculation.
- **Block II — Design your helicene.** Tell the notebook what kind of spectrum you want (e.g. "as strong a positive band as possible") and let a genetic algorithm search a space of **billions** of possible substitution patterns to find the best candidates.

Everything here is built on top of two neural network models trained on DFT/xtb-stda-computed ECD spectra of [6]helicene derivatives:

| Model | Input | What it knows about each substituent |
|---|---|---|
| **E2E_H** | 16 numbers | Only the Hammett σ constant per position (simplest, fastest, good baseline) |
| **E2E_64** | 64 numbers | Hammett σ, van der Waals volume, and the resonance parameters R+ and R− per position (richer, generally more accurate) |

Both predict the full ECD spectrum (100 points, 250–600 nm) in a single forward pass — no quantum chemistry involved at prediction time.

**You do not need to understand any machine learning to use this notebook.** Every code cell below is ready to run as-is; just change the few values marked `# <-- EDIT THIS`.


## ⚠️ Before you start: what you need

1. **A trained model package** for `E2E_H` and/or `E2E_64` — this is the `.zip` that `train_pipeline.py` produces (contains the `.keras` model, the scalers, `WL_MODEL.npy`, `feature_columns.json`, and `applicability_domain.joblib`). Upload it to this Colab session or keep it on your Google Drive.
2. That's it for **Block I**.
3. For **Block II** (the genetic algorithm), you also need `screen_substituents.py` in the `scripts/` folder — **this file is not bundled with this package yet** (see the warning in Block II for exactly what it needs to provide). Everything else needed for Block II is already included.


---
## Setup

Run these two cells once per Colab session.


In [ ]:
# If you're running this in Google Colab, mount your Drive so you can
# reach your model package and (optionally) save results there.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Colab — skipping Drive mount.")


In [ ]:
# Make sure the bundled scripts are importable, and install anything missing.
import sys, subprocess

SCRIPTS_DIR = "/content/HelicAI/scripts"   # <-- EDIT THIS if you placed the package somewhere else
sys.path.append(SCRIPTS_DIR)

for pkg in ["tensorflow", "scikit-learn", "joblib", "pandas", "matplotlib"]:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

print("Setup OK. Scripts folder on path:", SCRIPTS_DIR)


---
# Block I — Predict your helicene

## How this works, briefly

Each of the 16 positions on the [6]helicene core either carries a substituent or is unsubstituted (plain H). For each position, the model needs a small set of numbers describing the electronic and steric character of whatever sits there — **not** the raw chemical structure (no atoms, no bonds, no 3D coordinates go into the model).

- **E2E_H** only needs the **Hammett σ constant** of each substituent (16 numbers total).
- **E2E_64** needs **four** numbers per substituent — Hammett σ, van der Waals volume, and the resonance parameters R+ and R− (64 numbers total).

This is exactly why you can ask the model about a substituent it never saw in training: as long as you can give it that substituent's Hammett σ (for E2E_H) or all four descriptors (for E2E_64), it will happily predict a spectrum. Whether that prediction is trustworthy is a separate question — see the **applicability domain** section below.

The model outputs a fixed grid of 100 points between 250 and 600 nm — the same grid, and the same Gaussian-broadening convention (σ=0.30 eV), used to build the training data. It does not predict individual electronic transitions; it predicts the already-broadened curve directly, in one shot.


In [ ]:
from predict_ecd import load_model_package, predict_ecd
import json
import numpy as np
import matplotlib.pyplot as plt

# --- EDIT THIS: where your unzipped model package lives ---------------------
MODEL_DIR = "/content/drive/MyDrive/ecd_project/ml_experiments/models/E2E_64"
LABEL = "E2E_64"   # "E2E_64" or "E2E_H"
# ------------------------------------------------------------------------------

package = load_model_package(MODEL_DIR, LABEL)
print(f"Loaded {LABEL}. Expects {len(package.feature_columns)} descriptors: {package.feature_columns}")


## 1.1 Predict a compound using substituents already in your library

`data/substituent_library.json` has 16 known substituents (the ones the model actually trained on). Describe your molecule as a simple `{position: name}` dictionary — plain English names like `"bromo"`, `"NO2"`, `"CH3"` — and the helper below builds the right descriptor vector automatically, whichever model you loaded.


In [ ]:
with open("/content/HelicAI/data/substituent_library.json") as f:
    KNOWN_LIBRARY = json.load(f)

N_POSITIONS = 16

def build_descriptor_vector(compound: dict, label: str, known_library: dict,
                             new_substituents: dict = None) -> np.ndarray:
    """Turns a {position: substituent_name} dict into the exact descriptor
    vector a model expects.

    compound          : e.g. {2: "bromo", 15: "bromo"}. Unlisted positions
                         are treated as unsubstituted (plain H).
    label              : "E2E_H" (16 numbers) or "E2E_64" (64 numbers).
    known_library      : the loaded substituent_library.json (the 16
                         substituents the model trained on).
    new_substituents   : OPTIONAL dict for substituents NOT in known_library,
                         e.g. {"CF3": {"hammett": 0.54, "vdw": 42.74,
                                        "r_plus": 0.23, "r_minus": 0.27}}.
                         For E2E_H you may omit vdw/r_plus/r_minus entirely
                         -- only "hammett" is actually used.
    """
    names_to_key = {v.lower(): k for k, v in known_library["names"].items()}
    new_substituents = new_substituents or {}

    hammett = np.zeros(N_POSITIONS)
    vdw = np.full(N_POSITIONS, known_library["defaults"]["vdw"])
    r_plus = np.zeros(N_POSITIONS)
    r_minus = np.zeros(N_POSITIONS)

    for pos, name in compound.items():
        i = pos - 1
        low = name.lower()
        if low in names_to_key:
            key = names_to_key[low]
            hammett[i] = float(key)
            vdw[i] = known_library["descriptors"]["vdw"][key]
            r_plus[i] = known_library["descriptors"]["r_plus"][key]
            r_minus[i] = known_library["descriptors"]["r_minus"][key]
        elif name in new_substituents:
            d = new_substituents[name]
            hammett[i] = d["hammett"]
            vdw[i] = d.get("vdw", known_library["defaults"]["vdw"])
            r_plus[i] = d.get("r_plus", 0.0)
            r_minus[i] = d.get("r_minus", 0.0)
        else:
            raise ValueError(
                f"'{name}' is not in the known library and wasn't supplied via "
                f"new_substituents. Known names: {sorted(known_library['names'].values())}"
            )

    if label == "E2E_H":
        return hammett
    return np.concatenate([hammett, vdw, r_plus, r_minus])


def plot_prediction(wavelengths, spectrum, ad_info, title):
    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.plot(wavelengths, spectrum, linewidth=2.5, color="#C0272D")
    ax.axhline(0, color="grey", linewidth=0.8)
    ax.set_xlim(wavelengths[0], wavelengths[-1])
    ax.set_xlabel(r"$\lambda$ / nm", fontsize=16)
    ax.set_ylabel(r"$R\cdot10^{40}$ / esu cm erg G$^{-1}$", fontsize=16)
    flag = "  \u26a0 outside applicability domain" if ad_info["outside_reference_domain"] else ""
    ax.set_title(title + flag, fontsize=15, fontweight="bold")
    ax.tick_params(labelsize=12)
    ax.grid(alpha=0.2)
    fig.tight_layout()
    plt.show()

print("Helper functions ready.")


In [ ]:
# --- EDIT THIS: your compound, {position: substituent name} ----------------
compound = {
    2: "bromo",
    15: "bromo",
}
# ------------------------------------------------------------------------------

x = build_descriptor_vector(compound, LABEL, KNOWN_LIBRARY)
wavelengths, spectrum, ad_info = predict_ecd(x, package)

print("Compound:", ", ".join(f"Pos{p}={name}" for p, name in compound.items()))
print("Outside applicability domain?", ad_info["outside_reference_domain"])
dist = ad_info["nearest_training_distance"]
thresh = ad_info["threshold"]
print(f"  (distance to nearest training molecule: {dist:.3f}, reference threshold: {thresh:.3f})")

plot_prediction(wavelengths, spectrum, ad_info, f"{LABEL} — predicted ECD spectrum")


## 1.2 Predict a compound with a substituent the model has NEVER seen

This is the generalization test — the model was never trained on this substituent's chemistry at all. All you need to supply is:

- **For E2E_H**: just the Hammett σ constant.
- **For E2E_64**: all four descriptors (Hammett σ, van der Waals volume, R+, R−).

The cell below shows both. `data/new_substituent_library_EXAMPLE.json` has four ready-made examples (CF3, an amide, an ethyl, an ethoxy) you can try immediately, or replace with your own substituent's values.


In [ ]:
with open("/content/HelicAI/data/new_substituent_library_EXAMPLE.json") as f:
    _example = json.load(f)

# Reshape that file into the {name: {hammett, vdw, r_plus, r_minus}} format
# build_descriptor_vector() expects:
NEW_SUBSTITUENTS = {}
for key, name in _example["names"].items():
    NEW_SUBSTITUENTS[name] = {
        "hammett": float(key),
        "vdw": _example["descriptors"]["vdw"][key],
        "r_plus": _example["descriptors"]["r_plus"][key],
        "r_minus": _example["descriptors"]["r_minus"][key],
    }

print("New substituents available:", list(NEW_SUBSTITUENTS.keys()))


In [ ]:
# Example A: E2E_64 with a brand-new substituent that needs all 4 descriptors
compound_new = {2: "CF3", 15: "CF3"}   # <-- EDIT THIS

x_new = build_descriptor_vector(compound_new, "E2E_64", KNOWN_LIBRARY, NEW_SUBSTITUENTS)
wl_new, spec_new, ad_new = predict_ecd(x_new, package)   # package must be the E2E_64 one
plot_prediction(wl_new, spec_new, ad_new, "E2E_64 — never-seen substituent (CF3)")


In [ ]:
# Example B: E2E_H only ever needs the Hammett value -- you can invent one
# on the spot, even without knowing vdw/R+/R- at all.
package_H = load_model_package(MODEL_DIR.replace("E2E_64", "E2E_H"), "E2E_H")   # <-- EDIT THIS path if needed

compound_hammett_only = {2: "mystery_substituent", 15: "mystery_substituent"}   # <-- EDIT THIS
x_h = build_descriptor_vector(
    compound_hammett_only, "E2E_H", KNOWN_LIBRARY,
    new_substituents={"mystery_substituent": {"hammett": 0.50}},   # <-- just give sigma
)
wl_h, spec_h, ad_h = predict_ecd(x_h, package_H)
plot_prediction(wl_h, spec_h, ad_h, "E2E_H — only the Hammett value is needed")


## 1.3 Reading the "outside applicability domain" warning

Every prediction above also checked whether your molecule is **close** to the molecules the model actually trained on. Concretely: the notebook measures the distance (in standardized descriptor space) from your molecule to its single nearest neighbor in the training set, and compares that to a reference threshold computed from the training set itself.

- **Not flagged**: your molecule is about as "familiar" to the model as a typical training molecule is to its own neighbors.
- **⚠ Flagged**: your molecule sits further from anything the model trained on than is typical — the model is extrapolating. The prediction is still a number, but treat it with more caution, especially for never-seen substituents with Hammett/VdW/R values far outside the training range.

This is a **heuristic reliability flag, not a hard cutoff** — it doesn't block the prediction, it just tells you how much to trust it.


---
# Block II — Design your helicene

## ⚠️ Before running this block

`design_candidates.py` needs a file called **`screen_substituents.py`** in the `scripts/` folder, which is **not included in this package yet**. If you get an `ImportError` below, that's why — add the file and re-run.

For reference, `screen_substituents.py` must provide:
- `N_POSITIONS` (=16) and `ALL_64_COLS` (the ordered list of 64 feature names)
- `load_library(path)` — loads a substituent_library.json-style dictionary
- `build_vocabulary(library)` — returns the list of usable substituent keys
- `build_descriptor_matrix(candidates, library)` — turns a list of (positions, substituents) candidates into a descriptor matrix
- `is_canonical(positions, substituents)` / `mirror_assignment(positions, substituents)` — handle the helicene's own left-right (C2) symmetry, so the search doesn't waste time evaluating the same molecule twice under two different mirror-labelings

## How this works, briefly

`design_candidates.py` offers three search strategies, all built on top of the same E2E_64 model from Block I:

| Mode | What it does | Best for |
|---|---|---|
| `random` | Generates many random molecules and ranks them | A quick first look at the landscape |
| `hill_climb` | Starts from random seeds, repeatedly keeps whichever small mutation improves the score | Small numbers of substituents (≲5), fast |
| `genetic` | A full population-based genetic algorithm — crossover + mutation + elitism | Large, entangled search spaces (6, 10, even all 16 positions at once) — this is what covers the "billions of combinations" space |

You tell it what to optimize for via an **objective** — e.g. `"positive_integral"` (the biggest possible positive band), or your own custom function. You can also restrict the search to specific substituents, specific positions, force left-right symmetry, or focus the objective on one wavelength window instead of the whole spectrum.


In [ ]:
from design_candidates import genetic_search, BUILTIN_OBJECTIVES

print("Built-in objectives you can optimize for:", list(BUILTIN_OBJECTIVES))


In [ ]:
# --- EDIT THIS: your search settings -----------------------------------------
results = genetic_search(
    model_dir=MODEL_DIR,              # the E2E_64 package from Block I
    label="E2E_64",
    library_path="/content/HelicAI/data/substituent_library.json",
    objective="positive_integral",     # maximize the positive-band area
    k_min=1, k_max=10,                 # search 1 to 10 substituents at once
    population_size=300,
    n_generations=80,
    output_dir="/content/drive/MyDrive/ecd_project/design_results",   # <-- EDIT THIS
    top_n=20,
    plot_top_n=6,
)
# -------------------------------------------------------------------------------

results.head(10)


## 2.1 Focusing the search on one region of the spectrum

Instead of optimizing the whole spectrum, you can ask for the best band **specifically** around a chosen wavelength — e.g. "the strongest positive band near 450 nm" instead of "the strongest positive band anywhere":


In [ ]:
results_windowed = genetic_search(
    model_dir=MODEL_DIR,
    label="E2E_64",
    library_path="/content/HelicAI/data/substituent_library.json",
    objective="positive_integral",
    target_nm=450,          # <-- EDIT THIS: the wavelength you care about
    window_nm=30,           # <-- how far from target_nm still counts
    k_min=1, k_max=10,
    population_size=300,
    n_generations=60,
    output_dir="/content/drive/MyDrive/ecd_project/design_results_450nm",
    top_n=20,
    plot_top_n=6,
)
results_windowed.head(10)


## 2.2 Restricting the search space

You rarely want *every* substituent at *every* position. All three search modes accept the same restriction arguments:

- `allowed_substituents=["Br", "F", "NO2", "CN"]` — only use these
- `allowed_positions=[1, 2, 15, 16]` — only substitute here
- `force_symmetric=True` — only consider molecules symmetric under position *i* ↔ 17−*i* (the helicene's own fjord-to-fjord mirror pairing)
- `ad_penalty_weight=1.0` — steer the search away from candidates outside the applicability domain, instead of only flagging them after the fact

```python
results = genetic_search(
    model_dir=MODEL_DIR, label="E2E_64",
    library_path="/content/HelicAI/data/substituent_library.json",
    objective="positive_integral",
    allowed_substituents=["Br", "F", "NO2", "CN"],
    allowed_positions=[1, 2, 15, 16],
    force_symmetric=True,
    ad_penalty_weight=1.0,
    k_min=2, k_max=4,
    population_size=200, n_generations=50,
)
```


---
## Where to go next

- **Every script** in `scripts/` has a full docstring at the top explaining exactly what it does and its own command-line usage — open any of them directly for more detail than this notebook covers.
- `characterize_model.py` — full model report (precision overview, quartile spectra, error analysis by substituent, error vs. molecular complexity).
- `validate_new_substituents.py` / `validate_new_substituents_progression.py` — rigorous generalization testing against real DFT spectra of never-seen substituents, across increasing substitution complexity.
- `baseline_comparison.py` — sanity-checks the neural network against a 1-nearest-neighbor and a plain linear baseline on the exact same data split, so you can show the extra complexity actually earns its keep.
- `band_area_scaling_study.py` / `spectrum_scan.py` — large-scale statistics and full-spectrum screening across whole substitution levels (not just a genetic search toward one objective).

### A note on honesty for reviewers
Every prediction in this notebook comes from a neural network trained to approximate quantum-chemistry-computed spectra (DFT / xtb-stda), not from a new quantum-chemistry calculation. Always check the applicability-domain flag, and where possible, validate standout candidates (especially ones from Block II's genetic search) against an independent method or, ideally, an experimental measurement before relying on them.
